# Analyse de data drift — scoring crédit

Un modèle de machine learning apprend une relation entre le profil d'un demandeur de crédit et son risque de défaut. Cette relation peut se dégrader dans le temps pour deux raisons distinctes :

- **Data drift** : le profil des demandeurs change (ex. revenus, ancienneté professionnelle) sans que la relation entre profil et risque n'ait changé. On peut le mesurer immédiatement, en comparant les données récentes à celles utilisées à l'entraînement.
- **Concept drift** : la relation elle-même change — un même profil devient plus ou moins risqué qu'avant (ex. en période de récession). Sa mesure nécessite de connaître l'issue réelle des prêts (remboursé ou non), disponible seulement des mois après la décision — donc pas mesurable en temps réel avec les données dont on dispose ici.

**Méthode** : on compare statistiquement, feature par feature, les demandes de crédit récentes à la population utilisée pour entraîner le modèle actuel (`champion`), puis on résume ce comparatif en une décision globale — drift détecté ou non (détail du calcul en section 4).

**Données** : la référence est `serving_50_features.parquet`, le dataset d'entraînement du modèle (dépôt `OC-Pret-a-Depenser-scoring-model`, Kaggle *Home Credit Default Risk*), téléchargée depuis le bucket privé par `scripts/download_drift_reference.py` — aucune ligne filtrée, un NaN y est une donnée métier légitime. La production simulée provient d'un run k6 (`scripts/k6/predict_load.js`) rejouant un scénario de **récession économique** (`scripts/generate_drift_fixtures.py`) dont l'intensité croît progressivement ; on analyse les 5 000 prédictions les plus récentes, la moitié à plus forte intensité, pour mettre le drift le plus nettement en évidence.

In [1]:
from pathlib import Path
import sys

from dotenv import load_dotenv

sys.path.insert(0, "..")
load_dotenv(Path("..") / ".env")

from scripts.drift_analysis import (  # noqa: E402
    DRIFT_SHARE_THRESHOLD,
    build_datasets,
    classification_summary,
    dataset_drift_test,
    drift_scores,
    load_reference,
    run_drift_report,
    sample_recent_production,
)

from api.common.config import get_settings  # noqa: E402

## 1. Référence

Chargement du Parquet téléchargé par `make download-drift-reference` — l'intégralité des 307 511 lignes, NaN compris.

In [2]:
REFERENCE_PATH = Path("../data/drift/reference/serving_50_features.parquet")

reference = load_reference(str(REFERENCE_PATH))
print(f"{len(reference)} lignes de référence (dataset d'entraînement complet)")
reference.head()

307511 lignes de référence (dataset d'entraînement complet)


,payment_credit_ratio,EXT_SOURCE_2,EXT_SOURCE_1,EXT_SOURCE_3,DAYS_BIRTH,AMT_ANNUITY,ORGANIZATION_TYPE,previous_approved_cnt_payment_mean,DAYS_EMPLOYED,DAYS_ID_PUBLISH,...,bureau_active_amt_credit_max_overdue_mean,bureau_amt_credit_max_overdue_mean,installment_amt_instalment_sum,installment_days_entry_payment_sum,pos_sk_dpd_def_mean,NAME_EDUCATION_TYPE,bureau_amt_credit_sum_sum,installment_amt_instalment_max,bureau_active_amt_credit_sum_mean,bureau_active_days_credit_update_mean
0,0.060749,0.262949,0.083037,0.139376,-9461,24700.5,Business Entity Type 3,24.000000,-637.0,-2120,...,40.5,1681.029,219625.695,-5993.0,0.0,Secondary / secondary special,865055.565,53093.745,240994.2825,-15.5
1,0.027598,0.622246,0.311267,NaN,-16765,35698.5,School,10.000000,-1188.0,-291,...,0.0,0.000,1618864.650,-34633.0,0.0,Higher education,1017400.500,560835.360,810000.0000,-43.0
2,0.050000,0.555912,NaN,0.729567,-19046,6750.0,Government,4.000000,-225.0,-2531,...,NaN,0.000,21288.465,-2285.0,0.0,Secondary / secondary special,189037.800,10573.965,NaN,NaN
3,0.094941,0.650442,NaN,NaN,-19005,29686.5,Business Entity Type 3,18.000000,-3039.0,-2437,...,NaN,NaN,1007153.415,-4346.0,0.0,Secondary / secondary special,NaN,691786.890,NaN,NaN
4,0.042623,0.322738,NaN,NaN,-19932,21865.5,Religion,20.666667,-3038.0,-3458,...,NaN,0.000,835985.340,-68128.0,0.0,Secondary / secondary special,146250.000,22678.785,NaN,NaN


## 2. Population de production

On retient les 5 000 prédictions les plus récentes de `prediction_events` — la moitié du run k6 la plus proche de l'intensité 1 du ramp, donc la plus démonstrative du scénario de récession. La sélection se fait par ordre chronologique plutôt que par une fenêtre de dates fixe, pour rester valide quel que soit le moment où le run a été rejoué.

In [3]:
production = sample_recent_production(get_settings().database_url)
production_features = production[reference.columns.tolist()]
print(f"{len(production_features)} prédictions de production retenues")
production_features.head()

5000 prédictions de production retenues


,payment_credit_ratio,EXT_SOURCE_2,EXT_SOURCE_1,EXT_SOURCE_3,DAYS_BIRTH,AMT_ANNUITY,ORGANIZATION_TYPE,previous_approved_cnt_payment_mean,DAYS_EMPLOYED,DAYS_ID_PUBLISH,...,bureau_active_amt_credit_max_overdue_mean,bureau_amt_credit_max_overdue_mean,installment_amt_instalment_sum,installment_days_entry_payment_sum,pos_sk_dpd_def_mean,NAME_EDUCATION_TYPE,bureau_amt_credit_sum_sum,installment_amt_instalment_max,bureau_active_amt_credit_sum_mean,bureau_active_days_credit_update_mean
0,0.070342,0.613140,0.838400,NaN,-23852,16011.0,Self-employed,22.411761,NaN,-4863,...,NaN,NaN,1095351.975,-79545.0,0.000000,Secondary / secondary special,NaN,264290.985,NaN,NaN
1,0.062519,0.604830,0.566166,0.689730,-16426,13500.0,Business Entity Type 3,4.798200,-3487.256526,-4544,...,2059.367687,7761.077858,27453.825,-8490.0,0.000000,Higher education,2.191539e+05,13273.155,56266.876688,-62.0
2,0.049813,0.611972,NaN,NaN,-23253,26883.0,XNA,13.154569,NaN,-4580,...,NaN,2587.655266,1696305.960,-272337.0,0.000000,Secondary / secondary special,1.589327e+05,147863.160,NaN,NaN
3,0.060180,0.428316,NaN,0.645950,-19514,43299.0,Industry: type 3,7.992999,-71.292889,-3048,...,0.000000,0.000000,118494.315,-37595.0,0.357393,Secondary / secondary special,1.551420e+06,15123.015,396840.121512,-94.0
4,0.066875,0.585638,NaN,0.659382,-21259,37129.5,Business Entity Type 2,9.592079,-7879.239604,-4322,...,NaN,NaN,271893.105,-1364.0,0.000000,Secondary / secondary special,2.757732e+05,30210.345,160362.029703,-464.0


## 3. Schéma Evidently

Catégoriel vs numérique déduit directement du dtype des colonnes de la référence — pas besoin de dupliquer le schéma Pydantic de l'API.

In [4]:
categorical_columns = reference.select_dtypes(include="object").columns.tolist()
numerical_columns = reference.select_dtypes(exclude="object").columns.tolist()
print(f"{len(categorical_columns)} colonnes catégorielles : {categorical_columns}")
print(f"{len(numerical_columns)} colonnes numériques")

reference_dataset, production_dataset = build_datasets(reference, production_features)

5 colonnes catégorielles : ['ORGANIZATION_TYPE', 'CODE_GENDER', 'OCCUPATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_EDUCATION_TYPE']
45 colonnes numériques


## 4. Méthodologie : comment Evidently détecte le drift

`DataDriftPreset()` compare chaque feature de la production à sa distribution de référence, colonne par colonne, puis résume le résultat au niveau du dataset. Le calcul se fait en deux étapes.

### 4.1 Score de drift par colonne

Evidently choisit automatiquement une méthode selon le type de la colonne et le volume de données ([doc](https://docs.evidentlyai.com/metrics/customize_data_drift)). Au-delà de 1 000 lignes (notre cas, 5 000 lignes) :

**Numérique à plus de 5 valeurs distinctes → distance de Wasserstein normée.** La distance de Wasserstein (« Earth Mover's Distance ») mesure le coût minimal pour transformer une distribution en l'autre :

$$W_1(P, Q) = \int_0^1 \left| F_P^{-1}(u) - F_Q^{-1}(u) \right| \, du$$

où $F_P^{-1}$ et $F_Q^{-1}$ sont les fonctions quantiles (inverses des fonctions de répartition) des deux distributions. Evidently la normalise en la divisant par l'écart-type de la distribution de référence, ce qui exprime le décalage en fraction d'un écart-type de référence — un score de 0.1 signifie un décalage moyen équivalent à un dixième de cet écart-type.

**Catégoriel, ou numérique à 5 valeurs distinctes ou moins → distance de Jensen-Shannon.** Basée sur la divergence de Kullback-Leibler symétrisée :

$$JS(P, Q) = \tfrac{1}{2} KL(P \| M) + \tfrac{1}{2} KL(Q \| M), \qquad M = \tfrac{1}{2}(P + Q), \qquad KL(P \| Q) = \sum_i P_i \log \frac{P_i}{Q_i}$$

Evidently rapporte $\sqrt{JS(P, Q)}$ (distance, bornée dans [0, 1]).

**Seuil par colonne** : 0.1 par défaut pour ces deux métriques de distance — une colonne est « driftée » au-delà.

*(En dessous de 1 000 lignes, Evidently bascule sur des tests statistiques plus adaptés aux petits échantillons : test de Kolmogorov-Smirnov — $D = \sup_x |F_{ref}(x) - F_{cur}(x)|$ — pour le numérique, test du χ² ou Z pour le catégoriel, drift déclaré si p-valeur ≤ 0.05. Non utilisé ici, mentionné pour la complétude.)*

### 4.2 Décision au niveau du dataset

`DriftedColumnsCount` compte le nombre et la part de colonnes driftées parmi les 50. Le dataset est déclaré en drift si cette part dépasse un seuil configurable, `drift_share` (0.5 par défaut dans Evidently — pensé comme seuil générique et conservateur, pas calibré pour un cas d'usage particulier).

Pour ce service de credit scoring, on retient un seuil plus bas, **`drift_share = 0.2`** : une dérive doit alerter avant de toucher la moitié des 50 colonnes, dont beaucoup ont un impact marginal sur la décision du modèle.

## 5. Rapport de drift

In [5]:
drift_result = run_drift_report(reference_dataset, production_dataset, DRIFT_SHARE_THRESHOLD)

REPORT_PATH = Path("reports/drift_report.html")
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
drift_result.save_html(str(REPORT_PATH))
print(f"Rapport interactif enregistré dans {REPORT_PATH}")

Rapport interactif enregistré dans reports/drift_report.html


In [6]:
drift_scores_df = drift_scores(drift_result)
dataset_test = dataset_drift_test(drift_result)
print(dataset_test["description"])
print(f"Test Evidently : {dataset_test['status'].name}")
drift_scores_df.head(22)

Share of Drifted Columns: Actual value 0.400 >= 0.200
Test Evidently : FAIL


,feature,drift_score
0,previous_application_credit_ratio_mean,2.127504
1,payment_credit_ratio,0.879417
2,EXT_SOURCE_3,0.771676
3,EXT_SOURCE_2,0.748014
4,annuity_income_ratio,0.719525
5,EXT_SOURCE_1,0.681954
6,previous_approved_cnt_payment_mean,0.547879
7,previous_cnt_payment_mean,0.524502
8,employment_birth_ratio,0.521077
9,DAYS_EMPLOYED,0.447986


## 6. Sortie du modèle sur cette population

La dérive des features ne dit rien en soi de la dérive de la relation profil → risque (concept drift, non mesurable ici faute de résultat réel des prêts). On observe malgré tout la sortie du modèle sur cette population, comme signal indirect à surveiller.

In [7]:
refusal_rate, mean_probability = classification_summary(production)
print(f"Taux de refus sur cette population : {refusal_rate:.1%}")
print(f"Probabilité de défaut moyenne prédite : {mean_probability:.1%}")

Taux de refus sur cette population : 40.4%
Probabilité de défaut moyenne prédite : 46.6%


## 7. Conclusion

**Décision : Dataset Drift détecté** — le test Evidently natif (`drift_share = 0.2`, section 5) échoue : 20 features sur 50 dépassent le seuil de dérive par colonne (part = 0.40). Les features concernées correspondent exactement à celles ciblées par le scénario de récession simulé (scores externes, ratios d'endettement, ancienneté professionnelle, stress bureau, retards de paiement), tandis que les features de contrôle non touchées (`DAYS_BIRTH`, `CODE_GENDER`, montants de crédit) restent au niveau du bruit — le signal est réel, pas un artefact statistique.

**Type de drift** : c'est un **data drift** — un changement de la distribution des features en entrée. On ne peut pas conclure à un concept drift, qui nécessiterait de connaître l'issue réelle des prêts. Le taux de refus et la probabilité de défaut moyenne prédite (section 6), nettement supérieurs à la normale, sont un signal indirect cohérent avec une dégradation réelle plutôt qu'un simple changement de population bénin — mais restent une hypothèse, pas une mesure.

**Que faire face à ce drift ?**
- **Si le modèle reste fiable sur ces nouveaux profils** (à vérifier dès que des résultats réels sont disponibles) : pas d'action immédiate, continuer à surveiller.
- **Si la performance se dégrade** (concept drift confirmé a posteriori) : recalibrer le seuil de décision ou réentraîner le modèle sur des données représentatives du nouveau contexte.
- **Dans l'intervalle** : une politique de repli plus prudente (seuil relevé, revue manuelle des cas proches du seuil) limite le risque le temps d'obtenir des résultats confirmés.